# Parda — Phase 5: the full redaction pipeline + end-to-end benchmark

Page image → OCR (Tesseract + EasyOCR) → text PII (fine-tuned GLiNER + rules, gap fill) + visual PII (YOLO)
→ black boxes → redacted PDF + audit log (what was redacted, never the PII text).

**Settings:** Accelerator **GPU T4** (x1 or x2) · Internet **On**
**Inputs (Add Input):** your **Parda** (Phase 1) notebook · **Datasets tab →** `celeba-dataset` (jessicali9530)
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN` (the models are loaded from your private HF repos)

Run the cells in order (~45 min, the benchmark is resumable if the session dies).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts, Tesseract (+Hindi, Kannada), EasyOCR, GLiNER, YOLO, PyMuPDF (for PDF input)
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh
!pip install -q gliner ultralytics pymupdf
!python -c "import gliner, ultralytics, fitz, torch; print('gliner', gliner.__version__, '| ultralytics', ultralytics.__version__, '| pymupdf', getattr(fitz, 'VersionBind', '?'), '| GPU:', torch.cuda.get_device_name(0))"


In [ ]:
# 3. Data, face photos, and your models on Hugging Face
from huggingface_hub import HfApi
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import FacePool, find_faces_dir
DATA = find_data()
FACES = find_faces_dir("/kaggle/input", exclude=[DATA])
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
GLINER_FT, YOLO_REPO = f"{HF_USER}/parda-gliner-v1", f"{HF_USER}/parda-yolo-v1"
GLINER_BASE = "urchade/gliner_multi_pii-v1"
print("DATA  =", DATA, "| images:", _n_images(DATA))
print("FACES =", FACES, "|", FacePool(FACES).sizes() if FACES else "none")
print("models:", GLINER_FT, "|", YOLO_REPO)
assert FACES, "Add Input -> Datasets tab -> 'celeba-dataset' (jessicali9530)"
W = "/kaggle/working"


In [ ]:
# 4. Tests (~1 min): every line ok (the PDF test runs here because PyMuPDF is installed)
!python tests/test_pipeline.py


In [ ]:
# 5. The 300 benchmark pages with real (held-out) faces, exactly as in Phase 4 (~1 min)
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!ls {W}/yolo_bench/images/bench | wc -l


In [ ]:
# 6. END-TO-END BENCHMARK (~30-40 min; OCR is cached, so re-running continues where it stopped)
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {GLINER_FT} --gliner_base {GLINER_BASE} --yolo {YOLO_REPO} --cache {W}/e2e_cache --out {W}/e2e_eval 2>&1 | grep -v Warning | tail -60
from IPython.display import Image as IPImage, Markdown, display
display(Markdown(open(f"{W}/e2e_eval/report_e2e.md").read()))


In [ ]:
# 7. DEMO: redact one page per language with the full pipeline, show before | after
import json
from PIL import Image
from parda.pipeline.redact import build_redactor, redact_file
red = build_redactor("tesseract,easyocr", GLINER_FT, 0.3, True, YOLO_REPO)
metas = [json.loads(l) for l in open(f"{W}/yolo_bench/bench/meta.jsonl")]
demo = {}
for m in metas:  # first ID card / form per language
    demo.setdefault(m["lang"], m)
os.makedirs(f"{W}/demo", exist_ok=True)
for lang, m in sorted(demo.items()):
    src = f"{W}/yolo_bench/images/bench/{m['file']}"
    pdf, counts = redact_file(src, red, f"{W}/demo", lang=lang, png=True)
    name = os.path.splitext(m["file"])[0]
    a, b = Image.open(src).convert("RGB"), Image.open(f"{W}/demo/{name}_p1_redacted.png")
    pair = Image.new("RGB", (a.width + b.width + 20, max(a.height, b.height)), "white")
    pair.paste(a, (0, 0)); pair.paste(b, (a.width + 20, 0))
    pair.save(f"{W}/demo/{lang}_before_after.jpg", quality=85)
    print(lang, m["doc_type"], dict(counts))
    display(IPImage(f"{W}/demo/{lang}_before_after.jpg", width=900))
print(json.dumps(json.load(open(f"{W}/demo/{name}_audit.json"))["pages"][0]["regions"][:3], indent=1))  # audit: no PII text


In [ ]:
# 8. Collect results to commit to GitHub (download the folder from the Output tab)
R = f"{W}/phase5_results"
!mkdir -p {R}
!cp {W}/e2e_eval/report_e2e.md {W}/e2e_eval/report_e2e.json {R}/
!cp {W}/demo/*_before_after.jpg {W}/demo/*_audit.json {R}/
!ls -la {R}
